# 7005 · 数据处理与 A/B 示例

来源为[原版 Colab](https://colab.research.google.com/drive/1RosZTY1G_UD2hi6a4I0DdmLfq0xRV3zv) 的清洗、分词、Fog 和 NRC 评分代码，调整了文件路径和单元格结构。请把原始 CSV 与 NRC v0.92 词典放在 Notebook 工作目录，或通过 `UPWORTHY_DATA_DIR` 指定目录。运行结果写到 `UPWORTHY_RUN_DIR`，默认 `./7005_notebook_outputs/`。

需要 Python、numpy、pandas、scipy、matplotlib 和 Jupyter。

## Load original input

In [ ]:
import os
from pathlib import Path
import re
import html
import unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path(os.environ.get("UPWORTHY_DATA_DIR", "."))
RUN_DIR = Path(os.environ.get("UPWORTHY_RUN_DIR", "7005_notebook_outputs"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA_DIR / "regression_df_update_06122024.csv"
NRC_PATH = DATA_DIR / "NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt"
for source in (RAW_PATH, NRC_PATH):
    if not source.is_file():
        raise FileNotFoundError(f"{source}. Put both source files in DATA_DIR or set UPWORTHY_DATA_DIR.")
keep = ["id","clickability_test_id","headline","created_at","test_week",
        "impressions","clicks","click_rate","word_count"]
df = pd.read_csv(RAW_PATH, usecols=keep, low_memory=False)[keep]
assert df.id.is_unique and df[["id","clickability_test_id","headline"]].notna().all().all()
assert (df.impressions > 0).all() and df.clicks.between(0,df.impressions).all()
np.testing.assert_allclose(df.clicks/df.impressions,df.click_rate,atol=1e-12)
df["created_at"] = pd.to_datetime(df.created_at, utc=True)
df["test_week"] = df.test_week.astype(int)
print(f"{len(df):,} source headlines in {df.clickability_test_id.nunique():,} experiments")

## Clean headlines and tokenize

In [ ]:
ABBR = r"(?:[A-Za-z]\.){2,}"
WORD = r"[^\W_]+(?:'[^\W_]+)*(?:[.,][0-9]+)*"
TOKEN_RE = re.compile(ABBR + "|" + WORD)
SCAN_RE = re.compile(ABBR + "|" + WORD + r"|[.,;:!?()]|(?<=\s)-+(?=\s)")
_TAG = re.compile(r"<[^>]+>")
_ABBR_RE = re.compile(ABBR)

def clean_headline(raw):
    s = "" if pd.isna(raw) else str(raw)
    s = _TAG.sub(" ", html.unescape(_TAG.sub(" ", s)))
    for a,b in (("‘","'"),("’","'"),("‛","'"),("`","'"),("´","'"),("“",'"'),("”",'"'),
                ("…","..."),("—"," - "),("–"," - "),("\u200b",""),("™",""),("®",""),("©","")):
        s = s.replace(a,b)
    s = unicodedata.normalize("NFKD",s).replace("⁄","/")
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.encode("ascii","ignore").decode()
    s = s.replace('"'," ").replace("*","").replace("#","")
    s = re.sub(r"(?<![A-Za-z])'|'(?![A-Za-z])"," ",s)
    s = re.sub(r"[^A-Za-z0-9\s\.\,\!\?\:\;\-\(\)\/\$%&']"," ",s)
    return re.sub(r"\s+"," ",s).strip()

def count_sentences(text):
    protected = _ABBR_RE.sub(lambda m:m.group(0).replace(".","\x01"),text)
    return max(1,sum(bool(re.search(r"[A-Za-z0-9]",p)) for p in re.split(r"[.!?]+(?=\s|$)",protected)))

def count_punct(text):
    t = re.sub(r"\d[.,]\d","",_ABBR_RE.sub("",text))
    return len(re.findall(r"[.,!?;:()]",t))

def text_stats(text):
    toks = TOKEN_RE.findall(text)
    low = [t.lower() for t in toks]
    return {"headline_clean":text.lower(),"tokens_str":" ".join(low),"n_words":len(toks),
            "n_alpha_words":sum(bool(re.search(r"[A-Za-z]",t)) for t in toks),
            "n_unique":len(set(low)),"n_chars":len(text),"n_punct":count_punct(text),
            "n_sentences":count_sentences(text)}

df["headline_cased"] = df.headline.map(clean_headline)
df = pd.concat([df,pd.DataFrame([text_stats(t) for t in df.headline_cased],index=df.index)],axis=1)
day = df.created_at.dt.normalize()
df["days_since_start"] = (day-day.min()).dt.days
assert df.n_words.gt(0).all()
print(f"Token-count agreement with source: {df.n_words.eq(df.word_count).mean():.2%}")

## Gunning Fog

In [ ]:
SYLLABLE_OVERRIDES = {
    "something":2,"sometimes":2,"somehow":2,"somewhere":2,"somebody":3,
    "every":3,"everything":3,"everybody":4,"everywhere":3,"everyday":3,
    "marriage":2,"awesome":2,"brilliant":2,"female":2,"business":2,
    "facebook":2,"unique":2,"lifetime":2,"safety":2,"goosebumps":2,
    "williams":2,"behavior":3,"familiar":3,"reality":4,"terrifying":4,
    "horrifying":4,"scientific":4,"appreciate":4,"marijuana":4,
    "homelessness":3,"realize":3,"realized":3,"realizes":3,"realizing":4,
    "realised":3,"realising":4,"illustrated":4,"create":2,"created":3,
    "creating":3,"creative":3,"creation":3,"creator":3,"react":2,
    "reaction":3,"reactions":3,"reacting":3,"reacted":3,"ideas":3,
    "areas":3,"happier":3,"scarier":3,"funnier":3,"easier":3,
    "earlier":3,"beyonce":3,"video":3,"videos":3,"hostile":2,
    "skateboard":2,"baseball":2,"liquor":2,"brilliantly":3,
    "reinforcing":4,"chihuahua":3,"chappelle":2,"anyone":3,
    "scientist":3,"scientists":3,"entire":3,"suicide":3,"inspired":3,
    "employees":3,"employee":3,"mcdonalds":3,"ruining":3,"ourselves":3,
    "struggling":3,"satire":3,"theater":3,"barrier":3,"whoever":3,
    "quietly":3,"healthier":3,"creators":3,"heroin":3,"cocaine":3,
    "massacre":3,"employer":3,"aliens":3,"poetry":3,"taxpayers":3,
    "troubling":3,"sarcasm":3,"michelle":2,"pipeline":2,"profile":2,
    "brilliance":2,"sidewalk":2,"hometown":2,"chocolate":2,"julia":2,"desperate":2,
}
ACRONYMS = set("NFL CNN CEO PSA PTSD NRA NSA NYC MLK NBA HIV MTV CIA FDA KKK NYPD DNA TSA IRS NHL NCAA SNL DMV LGBT LGBTQ USA OMG LOL ESPN NSFW FYI DIY WTF WWII GOP FBI NAACP LGBTQIA".split())
_VOWEL_GROUPS = re.compile(r"[aeiouy]+")

def count_syllables(word):
    w0 = re.sub(r"[^a-z]","",word.lower())
    if not w0:
        return 0
    if w0 in SYLLABLE_OVERRIDES:
        return SYLLABLE_OVERRIDES[w0]
    if len(w0)<=3:
        return 1
    w = re.sub(r"(?:(?:[bdfkmnprtvwjq]|(?<![cs])h)es|(?<![td])ed|[^laeiouy]e)$","",w0)
    w = re.sub(r"(?<=[^aeiouy])e(?=(?:ly|less|ment|ness|ful|some)$)","",w)
    w = re.sub(r"^y","",w)
    n = len(_VOWEL_GROUPS.findall(w))
    x = sum(len(re.findall(pattern,w)) for pattern in
            (r"(?<![cstg])ia",r"(?<![cstg])io(?!u|n)",r"(?<![cstg])iou",r"(?<![qg])ua",r"uo",r"yi"))
    x += sum(len(re.findall(pattern,w0)) for pattern in (r"(?<!t)ience",r"(?<![tcs])ient",r"iety"))
    x += bool(re.search(r"[aeou]ing$",w0))
    x += bool(re.search(r"[^aeiou]ism$",w0))
    x += bool(re.search(r"[^aeiou]ea$",w0) and w0 not in ("plea","flea"))
    x += w0.endswith("iest") and w0!="priest"
    return max(1,n+x)

def token_syllables(tok):
    if re.fullmatch(ABBR,tok):
        return len(re.findall(r"[A-Za-z]",tok))
    if tok in ACRONYMS:
        return len(tok)
    return count_syllables(re.sub(r"'(?:s|ll|re|ve|d|m|t)$","",tok))

def fog_components(text):
    toks = TOKEN_RE.findall(text)
    n_w = len(toks)
    if n_w==0:
        return np.nan,np.nan,np.nan
    hard = sum(token_syllables(t)>=3 for t in toks)
    return .4*(n_w/count_sentences(text)+100*hard/n_w),hard/n_w,hard

df[["fog_score","hard_share","n_hard"]] = [fog_components(t) for t in df.headline_cased]
display(df.fog_score.describe())

## NRC scoring and three alternative rules

In [ ]:
NEGATORS = set("not no never nothing nobody none neither nor nowhere cannot without hardly barely scarcely dont doesnt didnt cant wont isnt arent wasnt werent couldnt wouldnt shouldnt hasnt havent hadnt aint mustnt".split())
NEG_WINDOW = 3
_CLITIC = re.compile(r"'(?:s|ll|re|ve|d|m)$")
FALLBACK_EXCLUDE = set("clothes politics news series species lens bus gas focus status its this was has does goes yes less unless across address business process success congress progress james according feed passed passes".split())
FALLBACK_BAD_TARGETS = {"fee","jam","passe","sham","accord"}
DOMAIN_EXCLUDE = {"black","white","john","minimum"}

def load_nrc(path):
    pos,neg = set(),set()
    with open(path,encoding="utf-8") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields)==3 and fields[2].strip()=="1":
                if fields[1]=="positive":
                    pos.add(fields[0])
                elif fields[1]=="negative":
                    neg.add(fields[0])
    return pos-neg,neg-pos

def _inflection_candidates(w):
    c=[]
    if len(w)<4 or w in FALLBACK_EXCLUDE:
        return c
    if w.endswith(("ies","ied")):
        c.append(w[:-3]+"y")
    if w.endswith("ier"):
        c.append(w[:-3]+"y")
    if w.endswith("iest"):
        c.append(w[:-4]+"y")
    if w.endswith("ed"):
        c += [w[:-1],w[:-2]]
        if len(w)>5 and w[-3]==w[-4]:
            c.append(w[:-3])
    if w.endswith("ying"):
        c.append(w[:-4]+"ie")
    if w.endswith("ing"):
        stem=w[:-3]
        c += [stem+"e",stem]
        if len(stem)>2 and stem[-1]==stem[-2]:
            c.append(stem[:-1])
    if w.endswith("es"):
        c += [w[:-1],w[:-2]]
    if w.endswith("s") and not w.endswith("ss"):
        c.append(w[:-1])
    return [x for x in c if len(x)>=3 and x not in FALLBACK_BAD_TARGETS]

class Lexicon:
    def __init__(self,pos,neg,fallback=True):
        self.pos,self.neg,self.fallback=pos,neg,fallback
        self.cache,self.fallback_hits={},{}
    def lookup(self,w):
        if w in self.cache:
            return self.cache[w]
        res=(w in self.pos,w in self.neg)
        if not any(res) and self.fallback:
            for cand in _inflection_candidates(w):
                if cand in self.pos or cand in self.neg:
                    res=(cand in self.pos,cand in self.neg)
                    self.fallback_hits[w]=cand
                    break
        self.cache[w]=res
        return res

def score_sentiment(text,lex,rule="flip"):
    n_pos=n_neg=n_words=f_pos=f_neg=cues=0
    since=None
    for match in SCAN_RE.finditer(text):
        tok=match.group(0)
        if not TOKEN_RE.fullmatch(tok):
            since=None
            continue
        n_words+=1
        low=tok.lower()
        negated=since is not None and since<NEG_WINDOW
        is_cue=low in NEGATORS or low.endswith("n't")
        is_pos=is_neg=False
        if not is_cue and re.search(r"[a-z]",low) and not re.search(r"\d",low):
            is_pos,is_neg=lex.lookup(_CLITIC.sub("",low))
            if is_pos and is_neg:
                is_pos=is_neg=False
        if negated and (is_pos or is_neg):
            if rule=="flip":
                if is_pos:
                    n_neg+=1
                    f_pos+=1
                else:
                    n_pos+=1
                    f_neg+=1
        else:
            n_pos+=is_pos
            n_neg+=is_neg
        if is_cue:
            cues+=1
            since=0
        elif since is not None:
            since+=1
    return dict(n_pos=n_pos,n_neg=n_neg,n_words=n_words,f_pos=f_pos,f_neg=f_neg,cues=cues)

pos,neg=load_nrc(NRC_PATH)
assert pos and neg
lex=Lexicon(pos,neg)
for suffix,matcher,rule in [("",lex,"flip"),("_alt",lex,"drop"),
                            ("_exact",Lexicon(pos,neg,False),"flip"),
                            ("_core",Lexicon(pos-DOMAIN_EXCLUDE,neg-DOMAIN_EXCLUDE),"flip")]:
    scores=pd.DataFrame([score_sentiment(t,matcher,rule) for t in df.headline_cased],index=df.index)
    np.testing.assert_array_equal(scores.n_words,df.n_words)
    df[f"pos_prop_nrc{suffix}"]=scores.n_pos/df.n_words
    df[f"neg_prop_nrc{suffix}"]=scores.n_neg/df.n_words
    if not suffix:
        df[["n_negators","n_neg_flipped","n_pos_flipped"]]=scores[["cues","f_neg","f_pos"]].to_numpy()

## Export and verify scored input

In [ ]:
predictors=["neg_prop_nrc","pos_prop_nrc","fog_score","n_words","days_since_start"]
assert np.isfinite(df[predictors]).all().all()
df.to_csv(RUN_DIR/"upworthy_v2_scored.csv",index=False)
study=df[df.groupby("clickability_test_id").clickability_test_id.transform("size").ge(2)].copy()
print(f"Analysis sample: {len(study):,} headlines, {study.clickability_test_id.nunique():,} experiments")
print(f"Clicks: {study.clicks.sum():,} | Impressions: {study.impressions.sum():,}")
display(study[predictors].describe().round(4))

## PPT data · A/B example and distributions

In [ ]:
titles=["Here's How Much It Costs To Block A Bill That 91% Of Americans Support",
        "How Much Does It Cost To Make Congress Ignore Something 91% Of Americans Support?"]
pair=study[study.headline_cased.isin(titles) & study.impressions.isin([5831,5580])].sort_values("neg_prop_nrc")
if len(pair)!=2 or pair.clickability_test_id.nunique()!=1:
    eligible=study[study.groupby("clickability_test_id").id.transform("size").eq(2)]
    spread=eligible.groupby("clickability_test_id").neg_prop_nrc.agg(lambda s:s.max()-s.min())
    pair=eligible[eligible.clickability_test_id.eq(spread.idxmax())].sort_values("neg_prop_nrc")

display(pair[["headline_cased", "clicks", "impressions", "neg_prop_nrc", "fog_score"]]
        .assign(ctr=pair.clicks / pair.impressions))

sizes = study.groupby("clickability_test_id").size()
variant_counts = sizes.clip(upper=9).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8.2, 3.9), layout="constrained")
labels = [str(count) if count < 9 else "9+" for count in variant_counts.index]
shares = variant_counts / len(sizes) * 100
ax.bar(labels, shares, color=["#98273e" if n == 4 else "#cf9ba6" for n in variant_counts.index])
ax.set(title="Headline versions per A/B test",
       xlabel="Headline versions per experiment", ylabel="Share of experiments (%)")
fig.savefig(RUN_DIR / "10_ab_test_variant_distribution.png", dpi=180)
plt.show()
plt.close(fig)

negativity = 100 * study.neg_prop_nrc
positive = negativity[negativity.gt(0)]
boundaries = [0, 5, 10, 15, 20, 25, 30, np.inf]
counts = pd.cut(positive, boundaries).value_counts(sort=False)
bin_labels = ["(0, 5]", "(5, 10]", "(10, 15]", "(15, 20]", "(20, 25]", "(25, 30]", ">30"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
zero = negativity.eq(0).mean() * 100
axes[0].bar(["None", "At least one"], [zero, 100-zero], color=["#98273e", "#cf9ba6"])
axes[0].set(title="All headlines", ylabel="Share of headlines (%)")
axes[1].bar(bin_labels, counts / len(positive) * 100, color="#98273e")
axes[1].set(title="Headlines with negative words", xlabel="Negative-word share (%)",
            ylabel="Share within this group (%)")
axes[1].tick_params(axis="x", labelrotation=30)
fig.savefig(RUN_DIR / "11_negative_word_distribution.png", dpi=180)
plt.show()
plt.close(fig)